# 13_RealTime_Prediction_GFS.ipynb
Prototype for real-time prediction using GFS summary features and the trained multimodal model.

In [ ]:
!pip -q install torch pandas scikit-learn

import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from google.colab import files

print("Upload gfs_features.csv")
u=files.upload()
gfs=pd.read_csv(list(u.keys())[0])

print("Upload weather_transformer.pth")
u=files.upload()
weather_model_path=list(u.keys())[0]

print("Upload cnn_features.csv")
u=files.upload()
cnn=pd.read_csv(list(u.keys())[0])

print("Upload multimodal_model.pth")
u=files.upload()
fusion_model_path=list(u.keys())[0]

# ----- Prepare weather features -----
weather_cols=[
    "temperature_mean",
    "humidity_mean",
    "dew_point_mean",
    "pressure_mean",
    "u_wind_mean",
    "v_wind_mean",
    "precipitation_sum"
]

X=gfs[weather_cols].fillna(0)

# NOTE:
# Replace this with the scaler saved during training for best results.
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

class WeatherTransformer(nn.Module):
    def __init__(self,input_dim,d_model=512,nhead=8,layers=2):
        super().__init__()
        self.embed=nn.Linear(input_dim,d_model)
        enc=nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        self.tr=nn.TransformerEncoder(enc,layers)

    def forward(self,x):
        x=self.embed(x)
        x=x.unsqueeze(1)
        return self.tr(x).squeeze(1)

device="cuda" if torch.cuda.is_available() else "cpu"

wt=WeatherTransformer(input_dim=len(weather_cols)).to(device)
wt.load_state_dict(torch.load(weather_model_path,map_location=device),strict=False)
wt.eval()

with torch.no_grad():
    weather_feat=wt(torch.tensor(X_scaled,dtype=torch.float32).to(device)).cpu()

cnn_cols=[c for c in cnn.columns if c!="event_id"]
cnn_tensor=torch.tensor(cnn.iloc[[0]][cnn_cols].values,dtype=torch.float32)

class FusionModel(nn.Module):
    def __init__(self,dim=512):
        super().__init__()
        self.attn=nn.MultiheadAttention(dim,8,batch_first=True)
        self.head=nn.Sequential(
            nn.Linear(dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,2)
        )
    def forward(self,cnn,tr):
        out,_=self.attn(cnn.unsqueeze(1),tr.unsqueeze(1),tr.unsqueeze(1))
        return self.head(out.squeeze(1))

fusion=FusionModel().to(device)
fusion.load_state_dict(torch.load(fusion_model_path,map_location=device))
fusion.eval()

with torch.no_grad():
    logits=fusion(cnn_tensor.to(device),weather_feat.to(device))
    probs=torch.softmax(logits,dim=1).cpu().numpy()[0]

classes=["FLASH","HEAT"]

print("\n===== REAL-TIME PREDICTION =====")
print("City:",gfs.loc[0,"city"])
print(f"FLASH Probability : {probs[0]*100:.2f}%")
print(f"HEAT Probability  : {probs[1]*100:.2f}%")
print("Predicted:",classes[probs.argmax()])

print("\nIMPORTANT:")
print("This prototype uses the first CNN feature vector from cnn_features.csv.")
print("For a true real-time system, replace it with CNN features extracted")
print("from the latest satellite image for the selected location.")
